# 01 — Nettoyage des données INSEE CSS02 (2017–2024)

**Projet :** Parité femmes/hommes dans l'industrie française  
**Source :** INSEE, Enquête Emploi en continu — Tableau CSS02  
**Objectif :** Construire un DataFrame propre et unifié sur 8 ans

---

### ⚠️ Notes méthodologiques

**Rupture 2021 :** L'enquête Emploi a été entièrement rénovée en 2021. Le terme *actifs occupés* (2017–2020) devient *personnes en emploi* (2021–2024). Les données restent comparables sur `part_femmes`, mais une ligne de rupture sera tracée en 2021 dans les graphiques.

**Codes SEXE CSV 2019–2020 :** `1 = Hommes`, `2 = Femmes`, `3 = Ensemble` ⚠️ contre-intuitif, vérifié sur Agriculture (~29% femmes).

**Structure des fichiers :**

| Années | Format | Onglets/Colonnes | Col. effectif total |
|--------|--------|-----------------|--------------------|
| 2017–2018 | `.xls` → `.xlsx` | `'3'`=Ensemble, `'2'`=Femmes | col 1 |
| 2019–2020 | `.csv` sep=`;` | SEXE numérique + NAFG038UN + CSER_OCC | POPOCC |
| 2021–2022 | `.csv` sep=`;` | SEXE texte + NAFG038UN + PCS1 | POPEMPL |
| 2023–2024 | `.xlsx` | ENSEMBLE, FEMMES, HOMMES | col 15 (Total) |

In [20]:
!pip install xlrd

## 0. Imports et configuration

In [19]:
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Imports OK')
print(f'  RAW_DIR  : {RAW_DIR.resolve()}')
print(f'  PROC_DIR : {PROC_DIR.resolve()}')

✓ Imports OK
  RAW_DIR  : /Users/adelaidaretuerto/Documents/MBA/Projets/projet-parite-industrie-france/data/raw
  PROC_DIR : /Users/adelaidaretuerto/Documents/MBA/Projets/projet-parite-industrie-france/data/processed


## 1. Référentiels

In [2]:
SECTEUR_MAP = {
    'zz':'Ensemble','AZ':'Agriculture, sylviculture et pêche',
    'ET':'Industrie (total)','BZ':'Industries extractives',
    'CA':'Alimentation, boissons, tabac','CB':'Textile, habillement, cuir',
    'CC':'Bois, papier, imprimerie','CD':'Cokéfaction et raffinage',
    'CE':'Industrie chimique','CF':'Industrie pharmaceutique',
    'CG':'Caoutchouc, plastiques, minéraux','CH':'Métallurgie, produits métalliques',
    'CI':'Informatique, électronique, optique','CJ':'Équipements électriques',
    'CK':'Machines et équipements','CL':'Matériels de transport',
    'CM':'Autres industries manufacturières','DZ':'Électricité, gaz, vapeur',
    'EZ':'Eau, assainissement, déchets','FZ':'Construction',
    'GI':'Commerce, transports, hébergement','GZ':'Commerce, réparation automobiles',
    'HZ':'Transports et entreposage','IZ':'Hébergement et restauration',
    'JZ':'Information et communication','JA':'Édition, audiovisuel',
    'JB':'Télécommunications','JC':'Activités informatiques',
    'KZ':'Activités financières et assurance','LZ':'Activités immobilières',
    'MN':'Activités spécialisées et soutien',
    'MA':'Activités juridiques, conseil, ingénierie',
    'MB':'Recherche-développement','MC':'Autres activités spécialisées',
    'NZ':'Activités de services administratifs',
    'OQ':'Admin. publique, enseignement, santé',
    'OZ':'Administration publique','PZ':'Enseignement',
    'QA':'Santé humaine','QB':'Hébergement médico-social, action sociale',
    'RU':'Arts, spectacles, activités récréatives',
    'RZ':'Autres activités de services',
    'SZ':'Activités des ménages employeurs',
    'TZ':'Activités extra-territoriales',
    'UZ':'Non classé','XX':'Non renseigné','EV':'Tertiaire (total)',
}

LIBELLE_MAP = {
    'Ensemble':'Ensemble','Non renseigné':'Non renseigné',
    'Non spécifié':'Non renseigné','Total':'Ensemble',
    'Agriculture, sylviculture et pêche':'Agriculture, sylviculture et pêche',
    'Industrie':'Industrie (total)',
    'Industrie manufacturière, industries extractives et autres':'Industrie (total)',
    'Industries extractives':'Industries extractives',
    'Fabrication de denrées alimentaires, de boissons et de produits à base de tabac':'Alimentation, boissons, tabac',
    "Fabrication de textiles, industries de l'habillement, industrie du cuir et de la chaussure":'Textile, habillement, cuir',
    'Travail du bois, industries du papier et imprimerie':'Bois, papier, imprimerie',
    'Cokéfaction et raffinage':'Cokéfaction et raffinage',
    'Industrie chimique':'Industrie chimique',
    'Industrie pharmaceutique':'Industrie pharmaceutique',
    "Fabrication de produits en caoutchouc et en plastique ainsi que d'autres produits minéraux non métalliques":'Caoutchouc, plastiques, minéraux',
    "Métallurgie et fabrication de produits métalliques à l'exception des machines et des équipements":'Métallurgie, produits métalliques',
    'Fabrication de produits informatiques, électroniques et optiques':'Informatique, électronique, optique',
    "Fabrication d'équipements électriques":'Équipements électriques',
    'Fabrication de machines et équipements n.c.a.':'Machines et équipements',
    'Fabrication de matériels de transport':'Matériels de transport',
    "Autres industries manufacturières ; réparation et installation de machines et d'équipements":'Autres industries manufacturières',
    "Production et distribution d'électricité, de gaz, de vapeur et d'air conditionné":'Électricité, gaz, vapeur',
    "Production et distribution d'eau ; assainissement, gestion des déchets et dépollution":'Eau, assainissement, déchets',
    'Construction':'Construction','Tertiaire':'Tertiaire (total)',
    'Commerce de gros et de détail, transports, hébergement et restauration':'Commerce, transports, hébergement',
    "Commerce ; réparation d'automobiles et de motocycles":'Commerce, réparation automobiles',
    'Transports et entreposage':'Transports et entreposage',
    'Hébergement et restauration':'Hébergement et restauration',
    'Information et communication':'Information et communication',
    'Edition, audiovisuel et diffusion':'Édition, audiovisuel',
    'Édition, audiovisuel et diffusion':'Édition, audiovisuel',
    'Télécommunications':'Télécommunications',
    "Activités informatiques et services d'information":'Activités informatiques',
    "Programmation, conseil et autres activités informatiques ; services d'information":'Activités informatiques',
    "Activités financières et d'assurance":'Activités financières et assurance',
    'Activités immobilières':'Activités immobilières',
    "Activités spécialisées, scientifiques et techniques et activités de services administratifs et de soutien":'Activités spécialisées et soutien',
    "Activités juridiques, comptables, de gestion, d'architecture, d'ingénierie, de contrôle et d'analyses techniques":'Activités juridiques, conseil, ingénierie',
    'Recherche-développement scientifique':'Recherche-développement',
    "Publicité et études de marché\u00a0; autres activités spécialisées, scientifiques et techniques\u00a0; activités vétérinaires":'Autres activités spécialisées',
    'Autres activités spécialisées, scientifiques et techniques':'Autres activités spécialisées',
    'Activités de services administratifs et de soutien':'Activités de services administratifs',
    'Administration publique, enseignement, santé humaine et action sociale':'Admin. publique, enseignement, santé',
    'Administration publique, défense, enseignement, santé humaine et action sociale':'Admin. publique, enseignement, santé',
    'Administration publique':'Administration publique',
    'Enseignement':'Enseignement',
    'Santé humaine':'Santé humaine',
    'Activités pour la santé humaine':'Santé humaine',
    'Hébergement médico-social et action sociale':'Hébergement médico-social, action sociale',
    'Hébergement médico-social et social et action sociale sans hébergement':'Hébergement médico-social, action sociale',
    'Arts, spectacles et activités récréatives':'Arts, spectacles, activités récréatives',
    'Autres activités de services':'Autres activités de services',
    "Arts, divertissement et loisirs\u00a0; autres activités de services\u00a0; activités des ménages, des organismes et organisations extraterritoriaux":'Arts et autres services',
    "Activités des ménages en tant qu'employeurs ; activités indifférenciées des ménages en tant que producteurs de biens et services pour usage propre":'Activités des ménages employeurs',
    'Activités extra-territoriales':'Activités extra-territoriales',
}

SECTEURS_ANALYSE = [
    'Agriculture, sylviculture et pêche',
    'Industrie (total)',
    'Construction',
    'Commerce, transports, hébergement',
    'Transports et entreposage',
    'Information et communication',
    'Activités informatiques',
    'Activités financières et assurance',
    'Activités spécialisées et soutien',
    'Admin. publique, enseignement, santé',
    'Santé humaine',
    'Hébergement médico-social, action sociale',
]

print(f'✓ {len(SECTEURS_ANALYSE)} secteurs retenus pour l\'analyse')

✓ 12 secteurs retenus pour l'analyse


## 2. Fonctions de lecture (4 parsers)

In [14]:
def convert_xls_to_xlsx(xls_path):
    xlsx_path = PROC_DIR / xls_path.name.replace('.xls', '_converted.xlsx')

    if xlsx_path.exists():
        return xlsx_path

    try:
        #Conversión usando pandas (reemplaza soffice)
        sheets = pd.read_excel(xls_path, sheet_name=None)

        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
            for name, df in sheets.items():
                df.to_excel(writer, sheet_name=name, index=False)

        print(f"✓ Convertido: {xlsx_path.name}")

    except Exception as e:
        print(f" Fallo en conversión de {xls_path.name}: {e}")
        raise

    return xlsx_path


def parse_xlsx_2017_2018(path, year):
    try:
        def read_sheet(s):
            try:
                df = pd.read_excel(path, sheet_name=s, header=None, engine='openpyxl')
                d = df.iloc[6:, [0,1]].copy()
                d.columns = ['secteur','effectif']
                d['secteur'] = d['secteur'].astype(str).str.strip()
                d = d[d['secteur'].notna() & ~d['secteur'].isin(['','nan'])]
                d['effectif'] = pd.to_numeric(d['effectif'].astype(str).str.replace(',','.'), errors='coerce')
                return d.dropna(subset=['effectif']).reset_index(drop=True)
            except Exception as e:
                print(f" Error leyendo hoja {s} ({year}): {e}")
                raise

        df_f, df_e = read_sheet('2'), read_sheet('3')
        n = min(len(df_f), len(df_e))

        r = df_e.iloc[:n][['secteur','effectif']].rename(columns={'effectif':'pop_total'}).copy()
        r['pop_femmes'] = df_f.iloc[:n]['effectif'].values
        r['annee'] = year
        r['part_femmes'] = (r['pop_femmes'] / r['pop_total'] * 100).round(1)

        return r[['annee','secteur','pop_femmes','pop_total','part_femmes']]

    except Exception as e:
        print(f" Error parsing XLSX 2017-2018 ({year}): {e}")
        raise


def parse_csv_2019_2020(path, year):
    try:
        df = pd.read_csv(path, sep=';', encoding='utf-8')

        df_f = df[(df['SEXE']==2) & (df['CSER_OCC']=='z')].copy()
        df_e = df[(df['SEXE']==3) & (df['CSER_OCC']=='z')].copy()

        df_f = df_f.rename(columns={'POPOCC':'pop_femmes','NAFG038UN':'code_secteur'})
        df_e = df_e.rename(columns={'POPOCC':'pop_total','NAFG038UN':'code_secteur'})

        m = df_e[['code_secteur','pop_total']].merge(
            df_f[['code_secteur','pop_femmes']], on='code_secteur'
        )

        m['secteur'] = m['code_secteur'].map(SECTEUR_MAP).fillna(m['code_secteur'])
        m['annee'] = year
        m['part_femmes'] = (m['pop_femmes'] / m['pop_total'] * 100).round(1)

        return m[['annee','secteur','pop_femmes','pop_total','part_femmes']]

    except Exception as e:
        print(f" Error parsing CSV 2019-2020 ({year}): {e}")
        raise


def parse_csv_2021_2022(path, year):
    try:
        df = pd.read_csv(path, sep=';', encoding='utf-8')

        df_f = df[(df['SEXE']=='FEMMES')   & (df['PCS1']=='z')].copy()
        df_e = df[(df['SEXE']=='ENSEMBLE') & (df['PCS1']=='z')].copy()

        for tmp in [df_f, df_e]:
            tmp['POPEMPL'] = pd.to_numeric(
                tmp['POPEMPL'].astype(str).str.replace(',','.'), errors='coerce'
            )

        df_f = df_f.rename(columns={'POPEMPL':'pop_femmes','NAFG038UN':'code_secteur'})
        df_e = df_e.rename(columns={'POPEMPL':'pop_total','NAFG038UN':'code_secteur'})

        m = df_e[['code_secteur','pop_total']].merge(
            df_f[['code_secteur','pop_femmes']], on='code_secteur'
        )

        m['secteur'] = m['code_secteur'].map(SECTEUR_MAP).fillna(m['code_secteur'])
        m['annee'] = year
        m['part_femmes'] = (m['pop_femmes'] / m['pop_total'] * 100).round(1)

        return m[['annee','secteur','pop_femmes','pop_total','part_femmes']]

    except Exception as e:
        print(f" Error parsing CSV 2021-2022 ({year}): {e}")
        raise


def parse_xlsx_2023_2024(path, year):
    try:
        def read_sheet(s):
            try:
                df = pd.read_excel(path, sheet_name=s, header=None, engine='openpyxl')
                d = df.iloc[8:, [0,15]].copy()
                d.columns = ['secteur','effectif']
                d['secteur'] = d['secteur'].astype(str).str.strip()
                d = d[d['secteur'].notna() & ~d['secteur'].isin(['','nan'])]
                d['effectif'] = pd.to_numeric(d['effectif'].astype(str).str.replace(',','.'), errors='coerce')
                return d.dropna(subset=['effectif']).reset_index(drop=True)
            except Exception as e:
                print(f" Error leyendo hoja {s} ({year}): {e}")
                raise

        df_f, df_e = read_sheet('FEMMES'), read_sheet('ENSEMBLE')

        m = df_e.merge(df_f, on='secteur', suffixes=('_total','_femmes'))
        m = m.rename(columns={'effectif_total':'pop_total','effectif_femmes':'pop_femmes'})

        m['annee'] = year
        m['part_femmes'] = (m['pop_femmes'] / m['pop_total'] * 100).round(1)

        return m[['annee','secteur','pop_femmes','pop_total','part_femmes']]

    except Exception as e:
        print(f" Error parsing XLSX 2023-2024 ({year}): {e}")
        raise


print('✓ 4 parsers définis')


✓ 4 parsers définis


## 3. Lecture de tous les fichiers

In [15]:
all_years = []

for year in [2017, 2018]:
    print(f'📂 {year} : .xls → .xlsx...')
    xlsx = convert_xls_to_xlsx(RAW_DIR / f'INSEE_CSS02_{year}.xls')
    print("Archivo generado:", xlsx)
    df = parse_xlsx_2017_2018(xlsx, year)
    all_years.append(df)
    print(f'   ✓ {len(df)} secteurs')

for year in [2019, 2020]:
    print(f'📂 {year} : CSV SEXE numérique...')
    df = parse_csv_2019_2020(RAW_DIR / f'INSEE_CSS02_{year}.csv', year)
    all_years.append(df)
    print(f'   ✓ {len(df)} secteurs')

for year in [2021, 2022]:
    print(f'📂 {year} : CSV SEXE texte...')
    df = parse_csv_2021_2022(RAW_DIR / f'INSEE_CSS02_{year}.csv', year)
    all_years.append(df)
    print(f'   ✓ {len(df)} secteurs')

for year in [2023, 2024]:
    print(f'📂 {year} : XLSX multi-onglets...')
    df = parse_xlsx_2023_2024(RAW_DIR / f'INSEE_CSS02_{year}.xlsx', year)
    all_years.append(df)
    print(f'   ✓ {len(df)} secteurs')

print('\n✅ Tous les fichiers chargés')

📂 2017 : .xls → .xlsx...
✓ Convertido: INSEE_CSS02_2017_converted.xlsx
Archivo generado: ../data/processed/INSEE_CSS02_2017_converted.xlsx
   ✓ 47 secteurs
📂 2018 : .xls → .xlsx...
✓ Convertido: INSEE_CSS02_2018_converted.xlsx
Archivo generado: ../data/processed/INSEE_CSS02_2018_converted.xlsx
   ✓ 47 secteurs
📂 2019 : CSV SEXE numérique...
   ✓ 47 secteurs
📂 2020 : CSV SEXE numérique...
   ✓ 47 secteurs
📂 2021 : CSV SEXE texte...
   ✓ 47 secteurs
📂 2022 : CSV SEXE texte...
   ✓ 47 secteurs
📂 2023 : XLSX multi-onglets...
   ✓ 47 secteurs
📂 2024 : XLSX multi-onglets...
   ✓ 47 secteurs

✅ Tous les fichiers chargés


/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 4. Concaténation, nettoyage et harmonisation

In [16]:
df_all = pd.concat(all_years, ignore_index=True)

df_all['annee']       = df_all['annee'].astype(int)
df_all['secteur']     = df_all['secteur'].astype(str).str.strip()
df_all['pop_femmes']  = pd.to_numeric(df_all['pop_femmes'], errors='coerce')
df_all['pop_total']   = pd.to_numeric(df_all['pop_total'],  errors='coerce')
df_all['part_femmes'] = pd.to_numeric(df_all['part_femmes'], errors='coerce')

df_all = df_all.dropna(subset=['part_femmes','pop_total'])
df_all = df_all[df_all['pop_total'] > 0]

# Harmoniser les libellés XLSX vers libellés courts unifiés
df_all['secteur'] = df_all['secteur'].replace(LIBELLE_MAP)

df_all = df_all.drop_duplicates(subset=['annee','secteur'], keep='first')
df_all = df_all.sort_values(['annee','secteur']).reset_index(drop=True)

print(f'Shape final : {df_all.shape}')
print(f'Années : {sorted(df_all["annee"].unique())}')
df_all.head(10)

Shape final : (374, 5)
Années : [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,annee,secteur,pop_femmes,pop_total,part_femmes
0,2017,Activités de services administratifs,512.6,1086.8,47.2
1,2017,Activités des ménages employeurs,256.2,288.8,88.7
2,2017,Activités extra-territoriales,8.8,19.0,46.3
3,2017,Activités financières et assurance,511.6,864.4,59.2
4,2017,Activités immobilières,187.8,393.9,47.7
5,2017,Activités informatiques,108.9,430.7,25.3
6,2017,"Activités juridiques, conseil, ingénierie",531.7,1157.8,45.9
7,2017,Activités spécialisées et soutien,1232.8,2649.5,46.5
8,2017,"Admin. publique, enseignement, santé",5742.1,8327.7,69.0
9,2017,Administration publique,1284.1,2433.0,52.8


## 5. Contrôles qualité

In [17]:
print('=== Contrôles qualité ===')

print(f'[1] part_femmes hors [0–100] : {len(df_all[~df_all["part_femmes"].between(0,100)])} ← attendu 0')
print(f'[2] Doublons annee×secteur   : {df_all.duplicated(["annee","secteur"]).sum()} ← attendu 0')
print(f'[3] NaN résiduels :\n{df_all.isnull().sum().to_string()}')

print('\n[4] Part femmes Ensemble par année (attendu ~47–52%) :')
print(df_all[df_all['secteur']=='Ensemble'][['annee','part_femmes']].to_string(index=False))

print('\n[5] Vérification secteurs clés :')
for s in ['Construction','Santé humaine','Industrie (total)','Agriculture, sylviculture et pêche']:
    vals = df_all[df_all['secteur']==s].set_index('annee')['part_femmes'].to_dict()
    print(f'  {s}: {vals}')

print('\n[6] Couverture 8 années (secteurs analyse) :')
for s in SECTEURS_ANALYSE:
    n = len(df_all[df_all['secteur']==s])
    print(f'  {"✓" if n==8 else f"⚠ {n}/8"}  {s}')

=== Contrôles qualité ===
[1] part_femmes hors [0–100] : 0 ← attendu 0
[2] Doublons annee×secteur   : 0 ← attendu 0
[3] NaN résiduels :
annee          0
secteur        0
pop_femmes     0
pop_total      0
part_femmes    0

[4] Part femmes Ensemble par année (attendu ~47–52%) :
 annee  part_femmes
  2017         48.1
  2018         48.3
  2019         48.5
  2020         48.6
  2021         48.9
  2022         48.9
  2023         49.0
  2024         48.8

[5] Vérification secteurs clés :
  Construction: {2017: 11.2, 2018: 11.5, 2019: 10.5, 2020: 11.1, 2021: 12.7, 2022: 12.8, 2023: 13.1, 2024: 12.8}
  Santé humaine: {2017: 74.9, 2018: 74.3, 2019: 74.3, 2020: 74.7, 2021: 75.5, 2022: 75.3, 2023: 75.3, 2024: 75.7}
  Industrie (total): {2017: 29.2, 2018: 28.9, 2019: 29.0, 2020: 28.5, 2021: 30.4, 2022: 31.1, 2023: 31.0, 2024: 30.7}
  Agriculture, sylviculture et pêche: {2017: 29.5, 2018: 27.0, 2019: 28.8, 2020: 29.7, 2021: 28.3, 2022: 29.7, 2023: 31.5, 2024: 29.5}

[6] Couverture 8 années (sec

## 6. Exports

In [18]:
df_complet = df_all.copy()
df_analyse = df_all[df_all['secteur'].isin(SECTEURS_ANALYSE)].copy()
df_pivot   = df_analyse.pivot_table(index='secteur', columns='annee', values='part_femmes').round(1)

df_complet.to_csv(PROC_DIR/'parite_complet_2017_2024.csv', index=False, sep=';', encoding='utf-8-sig')
df_analyse.to_csv(PROC_DIR/'parite_analyse_2017_2024.csv', index=False, sep=';', encoding='utf-8-sig')
df_pivot.to_csv(  PROC_DIR/'parite_pivot_part_femmes.csv',              sep=';', encoding='utf-8-sig')

print('✅ Exports :')
print(f'  parite_complet  : {df_complet.shape}')
print(f'  parite_analyse  : {df_analyse.shape}')
print(f'  parite_pivot    : {df_pivot.shape}')
print('\n=== Pivot % femmes (secteurs clés) ===')
print(df_pivot.to_string())

✅ Exports :
  parite_complet  : (374, 5)
  parite_analyse  : (96, 5)
  parite_pivot    : (12, 8)

=== Pivot % femmes (secteurs clés) ===
annee                                      2017  2018  2019  2020  2021  2022  2023  2024
secteur                                                                                  
Activités financières et assurance         59.2  58.0  56.6  55.8  56.9  56.5  56.8  55.5
Activités informatiques                    25.3  25.0  25.6  26.3  27.8  27.7  28.7  27.9
Activités spécialisées et soutien          46.5  46.9  47.8  47.7  47.4  47.8  49.1  49.4
Admin. publique, enseignement, santé       69.0  68.9  68.8  68.8  69.3  69.0  68.5  68.8
Agriculture, sylviculture et pêche         29.5  27.0  28.8  29.7  28.3  29.7  31.5  29.5
Commerce, transports, hébergement          41.3  41.9  42.0  42.2  41.8  42.2  41.8  41.9
Construction                               11.2  11.5  10.5  11.1  12.7  12.8  13.1  12.8
Hébergement médico-social, action sociale  83.6  83.7

## 7. Résumé

| Fichier | Description | Usage |
|---------|-------------|-------|
| `parite_complet_2017_2024.csv` | Tous secteurs, 2017–2024 | Exploration |
| `parite_analyse_2017_2024.csv` | 12 secteurs clés | **Graphiques** |
| `parite_pivot_part_femmes.csv` | Pivot secteur × année | **Tableau & Power BI** |

**Points méthodologiques README :**
- Rupture 2021 : tracer une ligne verticale dans les graphiques
- Codes SEXE CSV 2019/2020 : 1=Hommes, 2=Femmes (contre-intuitif, vérifié)
- Encodage exports : UTF-8-BOM pour compatibilité Windows/Excel

**→ Continuer avec `02_analysis_parite.ipynb`**